# Notebook 02 - Baseline Model Training

# BCI Competition IV, Dataset 2a

---

## Before running this notebook

This notebook expects **two Kaggle inputs** attached (Add Input on the right panel):

1. **The output of your preprocessing notebook** (the one that produced
   `preprocessed_dataset/A0X/A0XT.fif` / `A0XE.fif` per subject). On Kaggle, add it via
   *Add Input -> Notebooks -> (your preprocessing notebook)* — **make sure that notebook has
   been saved via "Save Version -> Save & Run All (Commit)"** first; just running it
   interactively does not produce output another notebook can read. Then pick the completed
   version when adding it here (not "Datasets" -- a notebook's output is a Notebook input).
2. **The raw BCI-IV 2a dataset** (or a `true_labels` bundle), if you want the *official*
   T -> E evaluation split. `*E.gdf`/`*E.fif` files mask every class cue as `"unknown"`
   (code 783) since labels were withheld for the original competition — the real evaluation
   labels are distributed **separately** as small `.mat` files (`A01E.mat` ... `A09E.mat`,
   each containing a `classlabel` array). If these aren't found, this notebook automatically
   falls back to an 80/20 split within each subject's T file instead (clearly labeled as
   non-official in the results).

Everything below auto-discovers files recursively under `/kaggle/input`, so it doesn't matter
what you named either input dataset.

## Three baselines, same protocol

All three models below follow the same per-subject T -> E evaluation protocol (subject-dependent,
matching how 2a results are conventionally reported):

1. **CSP + LDA** — single broadband CSP, classic fast baseline
2. **FBCSP + SVM** — multi-band CSP + mutual-information feature selection (the original
   2012 competition-winning approach)
3. **EEGNet** — compact end-to-end CNN (PyTorch, GPU-accelerated if enabled)

A final comparison table at the end summarizes accuracy/kappa across all three.

In [1]:
# Kaggle notebooks don't have mne preinstalled by default
!pip install -q mne

In [2]:
# ==========================================================
# Imports
# ==========================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import mne

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

In [3]:
# ==========================================================
# Configuration — auto-discover inputs
# ==========================================================

INPUT_ROOT = Path("/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline")
OUTPUT_PATH = Path("/kaggle/working")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Preprocessed recordings from the preprocessing notebook (*.fif)
fif_files = sorted(INPUT_ROOT.rglob("*.fif"))

# Anywhere a true-label .mat might live (raw dataset input, true_labels bundle, etc.)
mat_search_root = INPUT_ROOT

print(f"Found {len(fif_files)} preprocessed .fif file(s):")
for f in fif_files[:10]:
    print(" ", f)
if len(fif_files) > 10:
    print(f"  ... and {len(fif_files) - 10} more")

if not fif_files:

    print("\nNo .fif files found. Here's everything currently under /kaggle/input, "
          "to help diagnose what's actually attached:\n")

    if INPUT_ROOT.exists():
        for p in sorted(INPUT_ROOT.rglob("*")):
            print(" ", p)
    else:
        print("  /kaggle/input does not exist -- no inputs attached at all.")

    raise FileNotFoundError(
        "\nNo .fif files found under /kaggle/input. Checklist:\n"
        "  1. Did you SAVE the preprocessing notebook via 'Save Version -> Save & Run "
        "All (Commit)'? Just running cells interactively does not produce output another "
        "notebook can read.\n"
        "  2. Did the commit run finish successfully and produce files in its Output tab?\n"
        "  3. Did you add it here via 'Add Input -> Notebooks -> <your preprocessing "
        "notebook> -> (the completed version)'? (Not 'Datasets'.)\n"
        "  4. Check the directory listing printed just above -- if /kaggle/input has "
        "folders but no .fif files inside them, the commit likely didn't reach the "
        "batch-preprocessing cell."
    )

Found 18 preprocessed .fif file(s):
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A04/A04E.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A04/A04T.fif
  /kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A05/A05E.fif
  /kaggle/input/notebooks/sababahoque

In [4]:
# ==========================================================
# Class labels (2a's 4 native motor-imagery classes)
# ==========================================================

CUE_CODE_TO_LABEL = {
    "769": 0,   # left hand
    "770": 1,   # right hand
    "771": 2,   # foot
    "772": 3,   # tongue
}

CLASS_NAMES = {
    0: "Left Hand",
    1: "Right Hand",
    2: "Foot",
    3: "Tongue",
}

In [5]:
# ==========================================================
# Split preprocessed files into Training (T) vs Evaluation (E)
# ==========================================================

mi_files = [f for f in fif_files if f.stem.upper().endswith("T")]
eval_files = [f for f in fif_files if f.stem.upper().endswith("E")]

print(f"Training (labeled) recordings    : {len(mi_files)}")
print(f"Evaluation (unlabeled) recordings : {len(eval_files)}")

subjects_covered = sorted({f.stem[:-1] for f in mi_files})
print(f"\nSubjects with a training file: {subjects_covered}")

if len(subjects_covered) < 9:
    missing = sorted(set(f"A0{i}" for i in range(1, 10)) - set(subjects_covered))
    print(f"WARNING: missing training files for subjects: {missing}")

Training (labeled) recordings    : 9
Evaluation (unlabeled) recordings : 9

Subjects with a training file: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']


In [6]:
# ==========================================================
# Epoch extraction — Training (T) files
# ==========================================================

def extract_epochs(fif_file):
    """Extract labeled motor-imagery epochs from a *T file."""

    raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)

    cue_id = {k: v for k, v in event_id.items() if k in CUE_CODE_TO_LABEL}

    if not cue_id:
        raise ValueError(
            f"No motor-imagery cue codes (769-772) found in {fif_file.name}. "
            f"Available annotation codes: {list(event_id.keys())}"
        )

    epochs = mne.Epochs(
        raw, events, event_id=cue_id, tmin=-0.5, tmax=4.0,
        baseline=(-0.5, 0), preload=True, reject_by_annotation=True, verbose=False
    )

    X = epochs.get_data()

    inv_cue_id = {v: k for k, v in cue_id.items()}
    y = np.array([
        CUE_CODE_TO_LABEL[inv_cue_id[code]]
        for code in epochs.events[:, 2]
    ])

    return X, y

In [7]:
# ==========================================================
# Epoch extraction — Evaluation (E) files
# ==========================================================

def extract_epochs_eval(fif_file):
    """
    E files mark every MI cue as 'unknown' (783) rather than a real class
    code -- extract using that single code; true labels get attached
    separately from a .mat file located by find_true_labels().
    """

    raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)

    eval_cue_id = {k: v for k, v in event_id.items() if k == "783"}

    if not eval_cue_id:
        raise ValueError(
            f"No 'unknown' (783) cue markers found in {fif_file.name} -- "
            f"unexpected for a 2a evaluation file. Available codes: {list(event_id.keys())}"
        )

    epochs = mne.Epochs(
        raw, events, event_id=eval_cue_id, tmin=-0.5, tmax=4.0,
        baseline=(-0.5, 0), preload=True, reject_by_annotation=True, verbose=False
    )

    return epochs.get_data()

In [8]:
# ==========================================================
# Locate true evaluation labels (distributed separately from the GDF/FIF files)
# ==========================================================

def find_true_labels(subject_id, search_root):
    """
    Look for a separately-distributed true-label .mat file for this
    subject's evaluation set (conventionally '<subject_id>E.mat', e.g.
    'A01E.mat', containing a 'classlabel' array). Returns 0-indexed labels
    (0-3) matching CLASS_NAMES, or None if no file is found.
    """
    candidates = list(Path(search_root).rglob(f"{subject_id}E.mat"))

    if not candidates:
        return None

    mat = sio.loadmat(candidates[0])

    key = "classlabel" if "classlabel" in mat else next(
        (k for k in mat if not k.startswith("__")), None
    )

    if key is None:
        return None

    labels = np.array(mat[key]).squeeze().astype(int)

    return labels - 1  # 2a labels are 1-4 -> convert to 0-3

In [9]:
# ==========================================================
# Baseline — Per-Subject CSP + LDA (Official T -> E Split, with fallback)
# ==========================================================

results = []
fallback_used = []

for t_file in mi_files:

    subject_id = t_file.stem[:-1]  # strip trailing 'T'

    e_file = next((f for f in eval_files if f.stem[:-1] == subject_id), None)

    if e_file is None:
        print(f"[{subject_id}] Skipped -- no matching *E file found.")
        continue

    X_train, y_train = extract_epochs(t_file)

    true_labels = find_true_labels(subject_id, mat_search_root)

    used_fallback = False

    if true_labels is None:
        print(f"[{subject_id}] No true-label file (e.g. '{subject_id}E.mat') found -- "
              f"falling back to an 80/20 split within the T file (NOT the official split).")

        X_tr, X_te, y_tr, y_te = train_test_split(
            X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
        )

        X_train, y_train, X_test, y_test = X_tr, y_tr, X_te, y_te
        used_fallback = True

    else:

        X_test = extract_epochs_eval(e_file)

        if len(true_labels) != len(X_test):
            print(f"[{subject_id}] Skipped -- label count ({len(true_labels)}) != "
                  f"epoch count ({len(X_test)}); check for dropped/rejected epochs.")
            continue

        y_test = true_labels

    csp = CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)
    lda = LinearDiscriminantAnalysis()
    clf = Pipeline([("CSP", csp), ("LDA", lda)])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)

    results.append({
        "Subject": subject_id,
        "Train_Trials": len(y_train),
        "Test_Trials": len(y_test),
        "Accuracy": acc,
        "Kappa": kappa,
        "Split": "Fallback (80/20 within T)" if used_fallback else "Official (T -> E)"
    })

    fallback_used.append(used_fallback)

    print(f"[{subject_id}] acc={acc:.3f}  kappa={kappa:.3f}  "
          f"({'fallback split' if used_fallback else 'official split'})")

results_df = pd.DataFrame(results)

/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A01] No true-label file (e.g. 'A01E.mat') found -- falling back to an 80/20 split within the T file (NOT the official split).
Computing rank from data with rank=None
    Using tolerance 7e-05 (2.2e-16 eps * 25 dim * 1.3e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A01] acc=0.552  kappa=0.403  (fallback split)


/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A02] No true-label file (e.g. 'A02E.mat') found -- falling back to an 80/20 split within the T file (NOT the official split).
Computing rank from data with rank=None
    Using tolerance 4.3e-05 (2.2e-16 eps * 25 dim * 7.8e+09  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A02] acc=0.586  kappa=0.448  (fallback split)


/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


...

[A07] No true-label file (e.g. 'A07E.mat') found -- falling back to an 80/20 split within the T file (NOT the official split).
Computing rank from data with rank=None
    Using tolerance 5.5e-05 (2.2e-16 eps * 25 dim * 9.9e+09  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A07] acc=0.621  kappa=0.494  (fallback split)


/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A08/A08T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A08] No true-label file (e.g. 'A08E.mat') found -- falling back to an 80/20 split within the T file (NOT the official split).
Computing rank from data with rank=None
    Using tolerance 7e-05 (2.2e-16 eps * 25 dim * 1.3e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A08] acc=0.569  kappa=0.425  (fallback split)


/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A09/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A09] No true-label file (e.g. 'A09E.mat') found -- falling back to an 80/20 split within the T file (NOT the official split).
Computing rank from data with rank=None
    Using tolerance 7.8e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A09] acc=0.603  kappa=0.472  (fallback split)


In [10]:
# ==========================================================
# Summary
# ==========================================================

print("=" * 70)
print("CSP + LDA BASELINE -- SUMMARY")
print("=" * 70)

display(results_df)

if len(results_df):

    print(f"\nMean Accuracy : {results_df['Accuracy'].mean():.3f}  "
          f"(+/- {results_df['Accuracy'].std():.3f})")
    print(f"Mean Kappa    : {results_df['Kappa'].mean():.3f}  "
          f"(+/- {results_df['Kappa'].std():.3f})")

    if any(fallback_used):
        print("\nNOTE: one or more subjects used the FALLBACK split (no true-label .mat "
              "file was found for their evaluation set), so these numbers are not directly "
              "comparable to published official-split results for those subjects. Add the "
              "true_labels files as a Kaggle input to get the official T->E benchmark "
              "for every subject.")

    results_df.to_csv(OUTPUT_PATH / "csp_lda_baseline_results.csv", index=False)
    print(f"\nSaved to {OUTPUT_PATH / 'csp_lda_baseline_results.csv'}")

else:
    print("No subjects produced results -- check that mi_files/eval_files were found correctly, "
          "and that the preprocessing notebook's output was added as an input to this notebook.")

CSP + LDA BASELINE -- SUMMARY


,Subject,Train_Trials,Test_Trials,Accuracy,Kappa,Split
0,A01,230,58,0.551724,0.402772,Fallback (80/20 within T)
1,A02,230,58,0.586207,0.448276,Fallback (80/20 within T)
2,A03,230,58,0.706897,0.609968,Fallback (80/20 within T)
3,A04,230,58,0.379310,0.170441,Fallback (80/20 within T)
4,A05,230,58,0.362069,0.149425,Fallback (80/20 within T)
5,A06,230,58,0.465517,0.288203,Fallback (80/20 within T)
6,A07,230,58,0.620690,0.494253,Fallback (80/20 within T)
7,A08,230,58,0.568966,0.425287,Fallback (80/20 within T)
8,A09,230,58,0.603448,0.471892,Fallback (80/20 within T)



Mean Accuracy : 0.538  (+/- 0.114)
Mean Kappa    : 0.385  (+/- 0.153)

NOTE: one or more subjects used the FALLBACK split (no true-label .mat file was found for their evaluation set), so these numbers are not directly comparable to published official-split results for those subjects. Add the true_labels files as a Kaggle input to get the official T->E benchmark for every subject.

Saved to /kaggle/working/csp_lda_baseline_results.csv


---
## Baseline Model 2 — FBCSP + SVM (Filter-Bank CSP)

The original 2012 competition-winning approach: instead of one CSP fit on a single broad
band (1-40 Hz), CSP is fit separately in several narrow sub-bands (4 Hz wide, 4-40 Hz), and the
resulting log-variance features from all bands are concatenated. Mutual-information feature
selection then keeps the most discriminative features before classification (here, a linear SVM).

Same per-subject T -> E protocol as the CSP+LDA baseline above, with the same official/fallback
split logic.

In [11]:
# ==========================================================
# FBCSP — Band-filtered epoch extraction
# ==========================================================

FBCSP_BANDS = [(4,8),(8,12),(12,16),(16,20),(20,24),(24,28),(28,32),(32,36),(36,40)]
FBCSP_CSP_COMPONENTS = 4   # per band
FBCSP_TOP_K_FEATURES = 20  # after mutual-information selection across all bands

def extract_band_epochs(fif_file, cue_id_map, l_freq, h_freq, tmin=-0.5, tmax=4.0):
    """
    Re-filter a preprocessed recording into one narrow sub-band, then epoch it.
    cue_id_map: dict mapping annotation-code-string -> event integer to keep
    (use CUE_CODE_TO_LABEL for T files, or {'783': <code>} for E files).
    """

    raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)

    raw.filter(l_freq, h_freq, fir_design="firwin", verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)

    cue_id = {k: v for k, v in event_id.items() if k in cue_id_map}

    if not cue_id:
        raise ValueError(f"No matching cue codes found in {fif_file.name} for band "
                          f"{l_freq}-{h_freq} Hz.")

    epochs = mne.Epochs(
        raw, events, event_id=cue_id, tmin=tmin, tmax=tmax,
        baseline=(tmin, 0), preload=True, reject_by_annotation=True, verbose=False
    )

    return epochs, cue_id

In [12]:
# ==========================================================
# FBCSP — Feature extraction (multi-band CSP log-variance)
# ==========================================================

def fbcsp_features(t_file, e_file, y_train_ref, true_labels):
    """
    Fit one CSP per frequency band on the training epochs of that band, transform
    both train and test epochs through it, and concatenate log-variance features
    across all bands. CSP objects are refit per band (and per subject, since this
    whole function is called once per subject).
    """

    train_band_feats = []
    test_band_feats = []

    y_train = None

    for l_freq, h_freq in FBCSP_BANDS:

        train_epochs, cue_id = extract_band_epochs(t_file, CUE_CODE_TO_LABEL, l_freq, h_freq)

        X_train_band = train_epochs.get_data()

        inv_cue_id = {v: k for k, v in cue_id.items()}
        y_train = np.array([
            CUE_CODE_TO_LABEL[inv_cue_id[code]] for code in train_epochs.events[:, 2]
        ])

        if true_labels is not None:
            test_epochs, _ = extract_band_epochs(e_file, {"783": None}, l_freq, h_freq)
            X_test_band = test_epochs.get_data()
        else:
            X_test_band = None

        csp = CSP(n_components=FBCSP_CSP_COMPONENTS, reg="ledoit_wolf",
                  log=True, norm_trace=False)

        train_feat = csp.fit_transform(X_train_band, y_train)
        train_band_feats.append(train_feat)

        if X_test_band is not None:
            test_feat = csp.transform(X_test_band)
            test_band_feats.append(test_feat)

    X_train_feats = np.concatenate(train_band_feats, axis=1)

    X_test_feats = np.concatenate(test_band_feats, axis=1) if test_band_feats else None

    return X_train_feats, y_train, X_test_feats

In [13]:
# ==========================================================
# FBCSP — Per-Subject Training + Evaluation
# ==========================================================

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC

fbcsp_results = []
fbcsp_fallback_used = []

for t_file in mi_files:

    subject_id = t_file.stem[:-1]

    e_file = next((f for f in eval_files if f.stem[:-1] == subject_id), None)

    if e_file is None:
        print(f"[{subject_id}] Skipped -- no matching *E file found.")
        continue

    true_labels = find_true_labels(subject_id, mat_search_root)

    used_fallback = true_labels is None

    if used_fallback:

        print(f"[{subject_id}] No true-label file found -- FBCSP fallback: "
              f"80/20 split within the T file.")

        train_band_feats = []
        y_ref = None

        for l_freq, h_freq in FBCSP_BANDS:

            train_epochs, cue_id = extract_band_epochs(t_file, CUE_CODE_TO_LABEL, l_freq, h_freq)

            X_band = train_epochs.get_data()

            inv_cue_id = {v: k for k, v in cue_id.items()}
            y_ref = np.array([
                CUE_CODE_TO_LABEL[inv_cue_id[code]] for code in train_epochs.events[:, 2]
            ])

            csp = CSP(n_components=FBCSP_CSP_COMPONENTS, reg="ledoit_wolf",
                      log=True, norm_trace=False)

            feat = csp.fit_transform(X_band, y_ref)
            train_band_feats.append(feat)

        X_all_feats = np.concatenate(train_band_feats, axis=1)

        X_train_feats, X_test_feats, y_train, y_test = train_test_split(
            X_all_feats, y_ref, test_size=0.2, stratify=y_ref, random_state=SEED
        )

    else:

        X_train_feats, y_train, X_test_feats = fbcsp_features(
            t_file, e_file, None, true_labels
        )

        if len(true_labels) != len(X_test_feats):
            print(f"[{subject_id}] Skipped -- label/epoch count mismatch.")
            continue

        y_test = true_labels

    k = min(FBCSP_TOP_K_FEATURES, X_train_feats.shape[1])

    selector = SelectKBest(score_func=mutual_info_classif, k=k)
    X_train_sel = selector.fit_transform(X_train_feats, y_train)
    X_test_sel = selector.transform(X_test_feats)

    clf = SVC(kernel="linear", random_state=SEED)
    clf.fit(X_train_sel, y_train)
    y_pred = clf.predict(X_test_sel)

    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)

    fbcsp_results.append({
        "Subject": subject_id,
        "Train_Trials": len(y_train),
        "Test_Trials": len(y_test),
        "Accuracy": acc,
        "Kappa": kappa,
        "Split": "Fallback (80/20 within T)" if used_fallback else "Official (T -> E)"
    })

    fbcsp_fallback_used.append(used_fallback)

    print(f"[{subject_id}] FBCSP acc={acc:.3f}  kappa={kappa:.3f}  "
          f"({'fallback split' if used_fallback else 'official split'})")

fbcsp_results_df = pd.DataFrame(fbcsp_results)

[A01] No true-label file found -- FBCSP fallback: 80/20 split within the T file.


/tmp/ipykernel_16/1730665818.py:16: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


Computing rank from data with rank=None
    Using tolerance 7.6e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.


/tmp/ipykernel_16/1730665818.py:16: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


Computing rank from data with rank=None
    Using tolerance 7.6e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.


...

Computing rank from data with rank=None
    Using tolerance 7.6e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.


/tmp/ipykernel_16/1730665818.py:16: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A09/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


Computing rank from data with rank=None
    Using tolerance 7.6e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.


/tmp/ipykernel_16/1730665818.py:16: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A09/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


Computing rank from data with rank=None
    Using tolerance 7.6e-05 (2.2e-16 eps * 25 dim * 1.4e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
[A09] FBCSP acc=0.638  kappa=0.516  (fallback split)


In [14]:
# ==========================================================
# FBCSP — Summary
# ==========================================================

print("=" * 70)
print("FBCSP + SVM BASELINE -- SUMMARY")
print("=" * 70)

display(fbcsp_results_df)

if len(fbcsp_results_df):
    print(f"\nMean Accuracy : {fbcsp_results_df['Accuracy'].mean():.3f}  "
          f"(+/- {fbcsp_results_df['Accuracy'].std():.3f})")
    print(f"Mean Kappa    : {fbcsp_results_df['Kappa'].mean():.3f}  "
          f"(+/- {fbcsp_results_df['Kappa'].std():.3f})")

    fbcsp_results_df.to_csv(OUTPUT_PATH / "fbcsp_svm_baseline_results.csv", index=False)
else:
    print("No subjects produced results.")

FBCSP + SVM BASELINE -- SUMMARY


,Subject,Train_Trials,Test_Trials,Accuracy,Kappa,Split
0,A01,230,58,0.655172,0.541321,Fallback (80/20 within T)
1,A02,230,58,0.672414,0.563045,Fallback (80/20 within T)
2,A03,230,58,0.844828,0.793513,Fallback (80/20 within T)
3,A04,230,58,0.396552,0.197628,Fallback (80/20 within T)
4,A05,230,58,0.293103,0.058218,Fallback (80/20 within T)
5,A06,230,58,0.517241,0.356577,Fallback (80/20 within T)
6,A07,230,58,0.931034,0.908082,Fallback (80/20 within T)
7,A08,230,58,0.551724,0.401587,Fallback (80/20 within T)
8,A09,230,58,0.637931,0.516475,Fallback (80/20 within T)



Mean Accuracy : 0.611  (+/- 0.201)
Mean Kappa    : 0.482  (+/- 0.268)


---
## Baseline Model 3 — EEGNet (Compact CNN)

A compact convolutional network designed specifically for EEG (Lawhern et al., 2018): a temporal
convolution, a depthwise spatial convolution (one filter per channel, learning something CSP-like
but end-to-end), a separable convolution, then a linear classifier. Trained directly on the
preprocessed epochs (22 channels x time), no hand-crafted features.

Uses PyTorch (pre-installed on Kaggle, with GPU if you've enabled an accelerator for this
notebook). Same per-subject T -> E protocol as the other two baselines.

In [15]:
# ==========================================================
# EEGNet — Model Definition (PyTorch)
# ==========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class EEGNet(nn.Module):
    """
    Compact EEGNet (Lawhern et al. 2018), sized for BCI-IV 2a: 22 channels,
    4 classes, sampling rate passed in to size the temporal kernel.
    """

    def __init__(self, n_channels=22, n_classes=4, n_samples=1126, sfreq=250,
                 F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()

        half_sfreq = max(int(sfreq // 2), 1)

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, half_sfreq), padding=(0, half_sfreq // 2), bias=False),
            nn.BatchNorm2d(F1),

            # Depthwise spatial conv -- one set of spatial filters per channel
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        self.block2 = nn.Sequential(
            # Separable conv
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        # Infer the flattened feature size with a dummy forward pass
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.block2(self.block1(dummy))
            flat_size = out.numel()

        self.classifier = nn.Linear(flat_size, n_classes)

    def forward(self, x):
        # x: (batch, n_channels, n_samples) -> (batch, 1, n_channels, n_samples)
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

Using device: cpu


In [16]:
# ==========================================================
# EEGNet — Training / Evaluation Helper
# ==========================================================

from torch.utils.data import TensorDataset, DataLoader

def train_eegnet(X_train, y_train, X_test, y_test, sfreq=250,
                  n_epochs=100, batch_size=16, lr=1e-3):
    """
    Trains one EEGNet from scratch on a single subject's data and returns
    (accuracy, kappa) on the held-out test set.
    """

    # Per-trial z-score normalization (standard for EEGNet)
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8

    X_train_norm = (X_train - mean) / std
    X_test_norm = (X_test - mean) / std

    X_train_t = torch.tensor(X_train_norm, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t = torch.tensor(X_test_norm, dtype=torch.float32).to(DEVICE)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = EEGNet(
        n_channels=X_train.shape[1],
        n_classes=len(CLASS_NAMES),
        n_samples=X_train.shape[2],
        sfreq=sfreq
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(n_epochs):

        epoch_loss = 0.0

        for xb, yb in train_loader:

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * xb.size(0)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"    epoch {epoch+1}/{n_epochs}  loss={epoch_loss/len(train_ds):.4f}")

    model.eval()

    with torch.no_grad():
        preds = model(X_test_t).argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    kappa = cohen_kappa_score(y_test, preds)

    return acc, kappa

In [17]:
# ==========================================================
# EEGNet — Per-Subject Training + Evaluation
# ==========================================================

eegnet_results = []
eegnet_fallback_used = []

for t_file in mi_files:

    subject_id = t_file.stem[:-1]

    e_file = next((f for f in eval_files if f.stem[:-1] == subject_id), None)

    if e_file is None:
        print(f"[{subject_id}] Skipped -- no matching *E file found.")
        continue

    X_train, y_train = extract_epochs(t_file)

    true_labels = find_true_labels(subject_id, mat_search_root)

    used_fallback = False

    if true_labels is None:

        print(f"[{subject_id}] No true-label file found -- EEGNet fallback: "
              f"80/20 split within the T file.")

        X_tr, X_te, y_tr, y_te = train_test_split(
            X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
        )

        X_train, y_train, X_test, y_test = X_tr, y_tr, X_te, y_te
        used_fallback = True

    else:

        X_test = extract_epochs_eval(e_file)

        if len(true_labels) != len(X_test):
            print(f"[{subject_id}] Skipped -- label/epoch count mismatch.")
            continue

        y_test = true_labels

    print(f"[{subject_id}] Training EEGNet ({'fallback split' if used_fallback else 'official split'})...")

    acc, kappa = train_eegnet(X_train, y_train, X_test, y_test)

    eegnet_results.append({
        "Subject": subject_id,
        "Train_Trials": len(y_train),
        "Test_Trials": len(y_test),
        "Accuracy": acc,
        "Kappa": kappa,
        "Split": "Fallback (80/20 within T)" if used_fallback else "Official (T -> E)"
    })

    eegnet_fallback_used.append(used_fallback)

    print(f"[{subject_id}] EEGNet acc={acc:.3f}  kappa={kappa:.3f}\n")

eegnet_results_df = pd.DataFrame(eegnet_results)

/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A01/A01T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A01] No true-label file found -- EEGNet fallback: 80/20 split within the T file.
[A01] Training EEGNet (fallback split)...
    epoch 1/100  loss=1.4281
    epoch 20/100  loss=0.4936
    epoch 40/100  loss=0.1951
    epoch 60/100  loss=0.1278
    epoch 80/100  loss=0.1260
    epoch 100/100  loss=0.0677
[A01] EEGNet acc=0.655  kappa=0.540



/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A02/A02T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A02] No true-label file found -- EEGNet fallback: 80/20 split within the T file.
[A02] Training EEGNet (fallback split)...
    epoch 1/100  loss=1.4137
    epoch 20/100  loss=0.4729
    epoch 40/100  loss=0.2273
    epoch 60/100  loss=0.1672
    epoch 80/100  loss=0.1266
    epoch 100/100  loss=0.1313
[A02] EEGNet acc=0.621  kappa=0.494



/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A03/A03T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


...

[A07] No true-label file found -- EEGNet fallback: 80/20 split within the T file.
[A07] Training EEGNet (fallback split)...
    epoch 1/100  loss=1.3951
    epoch 20/100  loss=0.2615
    epoch 40/100  loss=0.0998
    epoch 60/100  loss=0.1052
    epoch 80/100  loss=0.0370
    epoch 100/100  loss=0.0542
[A07] EEGNet acc=0.793  kappa=0.724



/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A08/A08T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A08] No true-label file found -- EEGNet fallback: 80/20 split within the T file.
[A08] Training EEGNet (fallback split)...
    epoch 1/100  loss=1.4163
    epoch 20/100  loss=0.3027
    epoch 40/100  loss=0.1512
    epoch 60/100  loss=0.1107
    epoch 80/100  loss=0.0702
    epoch 100/100  loss=0.0539
[A08] EEGNet acc=0.897  kappa=0.862



/tmp/ipykernel_16/4139288277.py:8: RuntimeWarning: This filename (/kaggle/input/notebooks/sababahoquesaba/eeg-preprocessing-pipeline/preprocessed_dataset/A09/A09T.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)


[A09] No true-label file found -- EEGNet fallback: 80/20 split within the T file.
[A09] Training EEGNet (fallback split)...
    epoch 1/100  loss=1.4088
    epoch 20/100  loss=0.3775
    epoch 40/100  loss=0.1756
    epoch 60/100  loss=0.0814
    epoch 80/100  loss=0.0781
    epoch 100/100  loss=0.0681
[A09] EEGNet acc=0.810  kappa=0.747



In [18]:
# ==========================================================
# EEGNet — Summary
# ==========================================================

print("=" * 70)
print("EEGNET BASELINE -- SUMMARY")
print("=" * 70)

display(eegnet_results_df)

if len(eegnet_results_df):
    print(f"\nMean Accuracy : {eegnet_results_df['Accuracy'].mean():.3f}  "
          f"(+/- {eegnet_results_df['Accuracy'].std():.3f})")
    print(f"Mean Kappa    : {eegnet_results_df['Kappa'].mean():.3f}  "
          f"(+/- {eegnet_results_df['Kappa'].std():.3f})")

    eegnet_results_df.to_csv(OUTPUT_PATH / "eegnet_baseline_results.csv", index=False)
else:
    print("No subjects produced results.")

EEGNET BASELINE -- SUMMARY


,Subject,Train_Trials,Test_Trials,Accuracy,Kappa,Split
0,A01,230,58,0.655172,0.540412,Fallback (80/20 within T)
1,A02,230,58,0.620690,0.493651,Fallback (80/20 within T)
2,A03,230,58,0.844828,0.793431,Fallback (80/20 within T)
3,A04,230,58,0.620690,0.493450,Fallback (80/20 within T)
4,A05,230,58,0.862069,0.816238,Fallback (80/20 within T)
5,A06,230,58,0.603448,0.471055,Fallback (80/20 within T)
6,A07,230,58,0.793103,0.723919,Fallback (80/20 within T)
7,A08,230,58,0.896552,0.862014,Fallback (80/20 within T)
8,A09,230,58,0.810345,0.746725,Fallback (80/20 within T)



Mean Accuracy : 0.745  (+/- 0.118)
Mean Kappa    : 0.660  (+/- 0.158)


---
## Comparing All Three Baselines

In [19]:
# ==========================================================
# Final Comparison — CSP+LDA vs FBCSP+SVM vs EEGNet
# ==========================================================

comparison = pd.DataFrame({
    "Model": ["CSP + LDA", "FBCSP + SVM", "EEGNet"],
    "Mean_Accuracy": [
        results_df["Accuracy"].mean() if len(results_df) else np.nan,
        fbcsp_results_df["Accuracy"].mean() if len(fbcsp_results_df) else np.nan,
        eegnet_results_df["Accuracy"].mean() if len(eegnet_results_df) else np.nan,
    ],
    "Mean_Kappa": [
        results_df["Kappa"].mean() if len(results_df) else np.nan,
        fbcsp_results_df["Kappa"].mean() if len(fbcsp_results_df) else np.nan,
        eegnet_results_df["Kappa"].mean() if len(eegnet_results_df) else np.nan,
    ],
})

print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

display(comparison)

comparison.to_csv(OUTPUT_PATH / "baseline_comparison.csv", index=False)

BASELINE COMPARISON


,Model,Mean_Accuracy,Mean_Kappa
0,CSP + LDA,0.538314,0.384502
1,FBCSP + SVM,0.611111,0.481827
2,EEGNet,0.745211,0.660099
